In [ ]:
from pathlib import Path
import sys

# --- Resolve repo root (works whether you run from repo root or within notebooks/) ---
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    if (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    elif (REPO_ROOT.parent.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent.parent

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

from IPython.display import display
from notebooks.set_up import (
    ensure_dirs,
    SYNTH_DIR,
    TABLE_DIR, FIG_DIR,
    CATE_DIR, 
)

ensure_dirs()

# Inputs
output_result = str(CATE_DIR) + "/"

# Outputs
output_path = str(SYNTH_DIR) + "/"

Estimators and evaluation utilities live in `src/causalmix/cate/`.

In [ ]:
# CATE estimators + evaluation
from causalmix.cate import *


# Test of estimators on a single synthetic mcrpc dataset

In [0]:
# import mcrpc dataset
df_gen = spark.read.format("delta").load(output_path + "df_bias0_gen_0")
# convert dataset to pandas dataframe
df_gen = df_gen.toPandas()
print(df_gen)  # replaced display()
df_gen.info()

Abiraterone_prev,Enzalutamide_prev,anti_arr_pre,anti_diab_pre,any_infect_pre,arf_renal_pre,bld_thinner_pre,cvd_pre,dementia_pre,dm_pre,exp,hosp_ed_any,htn_pre,opiods_pre,race_cat,smoker_copd_pre,mets_site,trt_prev,age,Charlson
0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,1,Bone,Other,70,2
1,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,Bone,Taxane,78,3
1,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0,Bone,Taxane,77,3
0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,Bone,ADT_only,71,2
0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,Other,ADT_only,87,1
0,0,1,1,1,1,0,1,1,1,0,0,1,0,0,0,Other,ADT_only,84,6
1,0,1,0,0,0,0,1,1,0,0,0,1,0,0,1,Other,Other,84,2
0,0,1,0,1,0,0,0,0,0,1,0,1,1,0,0,Visceral,ADT_only,90,3
1,0,0,0,0,0,1,0,1,0,1,0,1,1,0,0,Bone,ARPI,90,1
0,0,0,0,1,1,0,0,1,0,0,0,0,1,0,1,Bone,ADT_only,76,4


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4098 entries, 0 to 4097
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Abiraterone_prev   4098 non-null   int64 
 1   Enzalutamide_prev  4098 non-null   int64 
 2   anti_arr_pre       4098 non-null   int64 
 3   anti_diab_pre      4098 non-null   int64 
 4   any_infect_pre     4098 non-null   int64 
 5   arf_renal_pre      4098 non-null   int64 
 6   bld_thinner_pre    4098 non-null   int64 
 7   cvd_pre            4098 non-null   int64 
 8   dementia_pre       4098 non-null   int64 
 9   dm_pre             4098 non-null   int64 
 10  exp                4098 non-null   int64 
 11  hosp_ed_any        4098 non-null   int64 
 12  htn_pre            4098 non-null   int64 
 13  opiods_pre         4098 non-null   int64 
 14  race_cat           4098 non-null   int64 
 15  smoker_copd_pre    4098 non-null   int64 
 16  mets_site          4098 non-null   object


In [0]:
# import truth 
true_ate = np.load(output_path + "ate_bias0_0.npy")
true_cate = np.load(output_path + "ite_bias0_0.npy")

In [0]:
numerical_var = ["age", "Charlson"]
print(numerical_var)
categorical_var = df_gen.select_dtypes(include=['object', 'category']).columns.to_list()
print(categorical_var)
binary_var = df_gen.columns.difference(categorical_var + numerical_var).to_list()
print(binary_var)
unique_levels = {col: df_gen[col].nunique() for col in categorical_var}
print(unique_levels)
X = df_gen.drop(["exp", "hosp_ed_any"], axis=1)
print(X.columns)

['age', 'Charlson']
['mets_site', 'trt_prev']
['Abiraterone_prev', 'Enzalutamide_prev', 'anti_arr_pre', 'anti_diab_pre', 'any_infect_pre', 'arf_renal_pre', 'bld_thinner_pre', 'cvd_pre', 'dementia_pre', 'dm_pre', 'exp', 'hosp_ed_any', 'htn_pre', 'opiods_pre', 'race_cat', 'smoker_copd_pre']
{'mets_site': 3, 'trt_prev': 4}
Index(['Abiraterone_prev', 'Enzalutamide_prev', 'anti_arr_pre',
       'anti_diab_pre', 'any_infect_pre', 'arf_renal_pre', 'bld_thinner_pre',
       'cvd_pre', 'dementia_pre', 'dm_pre', 'htn_pre', 'opiods_pre',
       'race_cat', 'smoker_copd_pre', 'mets_site', 'trt_prev', 'age',
       'Charlson'],
      dtype='object')


In [0]:
df = preprocess_data(
    df_gen,
    categorical_cols=categorical_var,
    continuous_cols=numerical_var,
    binary_cols=binary_var,
    dummy_code=False

)
df.info()
df[["age", "Charlson"]].describe()
Y = "hosp_ed_any"
T = "exp"

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4098 entries, 0 to 4097
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Abiraterone_prev    4098 non-null   float64
 1   Enzalutamide_prev   4098 non-null   float64
 2   anti_arr_pre        4098 non-null   float64
 3   anti_diab_pre       4098 non-null   float64
 4   any_infect_pre      4098 non-null   float64
 5   arf_renal_pre       4098 non-null   float64
 6   bld_thinner_pre     4098 non-null   float64
 7   cvd_pre             4098 non-null   float64
 8   dementia_pre        4098 non-null   float64
 9   dm_pre              4098 non-null   float64
 10  exp                 4098 non-null   float64
 11  hosp_ed_any         4098 non-null   float64
 12  htn_pre             4098 non-null   float64
 13  opiods_pre          4098 non-null   float64
 14  race_cat            4098 non-null   float64
 15  smoker_copd_pre     4098 non-null   float64
 16  age   

In [0]:
df2 = preprocess_data(
    df_gen,
    categorical_cols=categorical_var,
    continuous_cols=numerical_var,
    binary_cols=binary_var,
    dummy_code=True

)
df2.info()
df2[["age", "Charlson"]].describe()
Y = "hosp_ed_any"
T = "exp"

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4098 entries, 0 to 4097
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Abiraterone_prev    4098 non-null   float64
 1   Enzalutamide_prev   4098 non-null   float64
 2   anti_arr_pre        4098 non-null   float64
 3   anti_diab_pre       4098 non-null   float64
 4   any_infect_pre      4098 non-null   float64
 5   arf_renal_pre       4098 non-null   float64
 6   bld_thinner_pre     4098 non-null   float64
 7   cvd_pre             4098 non-null   float64
 8   dementia_pre        4098 non-null   float64
 9   dm_pre              4098 non-null   float64
 10  exp                 4098 non-null   float64
 11  hosp_ed_any         4098 non-null   float64
 12  htn_pre             4098 non-null   float64
 13  opiods_pre          4098 non-null   float64
 14  race_cat            4098 non-null   float64
 15  smoker_copd_pre     4098 non-null   float64
 16  age   

In [0]:
import pandas as pd
    
print("=" * 70)
print("TESTING quick-version CATE ESTIMATORS WITH MCRPC SYNTHETIC DATA")
print("=" * 70)

print(f"\nDataset shape: {df.shape}")
print(f"Treatment prevalence: {df[T].mean():.3f}")
print(f"Outcome prevalence: {df[Y].mean():.3f}")
print(f"True ATE: {true_ate.item():.4f}")
print(f"True CATE range: [{true_cate.min():.4f}, {true_cate.max():.4f}]")


print("\n" + "=" * 70)
print("RUNNING ESTIMATORS")
print("=" * 70)

TESTING quick-version CATE ESTIMATORS WITH MCRPC SYNTHETIC DATA

Dataset shape: (4098, 25)
Treatment prevalence: 0.556
Outcome prevalence: 0.174
True ATE: 0.0538
True CATE range: [-0.0068, 0.1478]

RUNNING ESTIMATORS


In [0]:
results_dmllinear = dml_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="linear",
        n_bootstrap=10,  # Using fewer for speed
        ci_level=0.95
    )
results_dmllasso = dml_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="lasso",
        n_bootstrap=10,  # Using fewer for speed
        ci_level=0.95
    )
results_dmlGbr = dml_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="GBR",
        n_bootstrap=10,  # Using fewer for speed
        ci_level=0.95
    )

In [0]:
results_dr = drlearner_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="GBR",
        n_bootstrap=10,  # Using fewer for speed
        ci_level=0.95
    )
# takes 30 mintues for the "auto" version

results_dr2 = drlearner_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="lasso",
        n_bootstrap=10,  # Using fewer for speed
        ci_level=0.95
    )
# takes 9 minutes for the "GBR" version

results_dr3 = drlearner_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="linear",
        n_bootstrap=10,  # Using fewer for speed
        ci_level=0.95
    )

In [0]:
results_xGbr = xlearner_binary(
    df2,
    outcome_col=Y,
    treatment_col=T,
    feature_cols=None,
    method="GBR",
    n_bootstrap=10,  # Using fewer for speed
    ci_level=0.95,
    random_state=42
)

results_xlasso = xlearner_binary(
    df2,
    outcome_col=Y,
    treatment_col=T,
    feature_cols=None,
    method="linear", # it is acutally ridge now after recoding
    n_bootstrap=10,  # Using fewer for speed
    ci_level=0.95,
    random_state=42
)

In [0]:
results_cf = causal_forest(df, outcome = Y, treatment = T, covariates=None, alpha=0.05, num_trees=100, min_node_size=5)
#results_cf2 = causal_forest(df2, outcome = Y, treatment = T, covariates=None, alpha=0.05, num_trees=100)
results_bcf = bayesian_causal_forest(
    df,
    outcome = Y,
    treatment = T,
    nburn=50,
    nsim=100,
    alpha=0.05,
    random_state=36
)
# nburn 1000, nsim 2000 took 40min
# nburn 500, nsim 1000 took 20min

In [0]:
eval_dr_Gbr = evaluate_estimator(
    results_dr,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="DRLearner-GBR"
)

eval_dr_lasso = evaluate_estimator(
    results_dr2,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="DRLearner-lasso"
)

eval_dr_linear = evaluate_estimator(
    results_dr3,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="DRLearner-linear"
)

eval_x_lasso = evaluate_estimator(
    results_xlasso,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="XLearner-lasso"
)

eval_x_Gbr = evaluate_estimator(
    results_xGbr,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="XLearner-GBR"
)

eval_dml_linear = evaluate_estimator(
    results_dmllinear,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="DML-linear"
)
eval_dml_lasso = evaluate_estimator(
    results_dmllasso,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="DML-lasso"
)

eval_dml_Gbr = evaluate_estimator(
    results_dmlGbr,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="DML-GBR"
)

eval_cf = evaluate_estimator(
    results_cf,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="CausalForest-one hot"
)

eval_bcf = evaluate_estimator(
    results_bcf,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="BayesianCausalForest"
)
# Compare
comparison_df = compare_estimators([eval_dr_Gbr, eval_dr_linear, eval_dr_lasso, eval_x_Gbr, eval_x_lasso, eval_dml_Gbr, eval_dml_lasso, eval_dml_linear, eval_cf, eval_bcf])
print("\n" + comparison_df.to_string(index=False))


           estimator  runtime_sec        ate_bias  ate_stderr  ate_coverage  cate_rmse  cate_coverage
       DRLearner-GBR          NaN  [0.0073779263]    0.116101             1   0.143988       0.608834
    DRLearner-linear          NaN  [0.0073779263]    0.042284             1   0.043628       0.953880
     DRLearner-lasso          NaN  [0.0073779263]    0.009104             1   0.022589       0.305271
        XLearner-GBR          NaN  [-0.021300148]    0.081923             1   0.099963       0.668619
      XLearner-lasso          NaN  [-0.026548054]    0.031682             1   0.034964       0.636164
             DML-GBR          NaN  [0.0020698644]    0.110430             1   0.132757       0.625183
           DML-lasso          NaN  [0.0024935864]    0.006203             1   0.021368       0.222792
          DML-linear          NaN   [0.002164945]    0.008533             1   0.041489       0.931186
CausalForest-one hot          NaN  [-0.001628492]    0.008940             1   0.0

In [0]:
import pandas as pd
    
print("=" * 70)
print("RUNNING ALL full-version CATE ESTIMATORS WITH MCRPC SYNTHETIC DATA")
print("=" * 70)

print(f"\nDataset shape: {df.shape}")
print(f"Treatment prevalence: {df[T].mean():.3f}")
print(f"Outcome prevalence: {df[Y].mean():.3f}")
print(f"True ATE: {true_ate.item():.4f}")
print(f"True CATE range: [{true_cate.min():.4f}, {true_cate.max():.4f}]")


print("\n" + "=" * 70)
print("RUNNING ESTIMATORS")
print("=" * 70)
    

start = time.perf_counter()
results_GbrXLearner = xlearner_binary(
    df2,
    outcome_col=Y,
    treatment_col=T,
    feature_cols=None,
    method="GBR",
    n_bootstrap=100,
    ci_level=0.95,
    random_state=42
)
results_GbrXLearner["runtime_sec"] = time.perf_counter() - start

# Evaluate performance
print("\n" + "=" * 70)
print("EVALUATION METRICS")
print("=" * 70)

eval_GbrXLearner = evaluate_estimator(
    results_GbrXLearner, 
    true_cate=true_cate, 
    true_ate=true_ate, 
    estimator_name="XLearner-GBR"
)

start = time.perf_counter()
results_LassoXLearner= xlearner_binary(
    df2,
    outcome_col=Y,
    treatment_col=T,
    feature_cols=None,
    method="linear",
    n_bootstrap=100,
    ci_level=0.95,
    random_state=42
)
results_LassoXLearner["runtime_sec"] = time.perf_counter() - start

eval_LassoXLearner = evaluate_estimator(
    results_LassoXLearner,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="XLearner-Lasso"
)
start = time.perf_counter()
results_GbrDML = dml_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR", # auto
        method_cate="GBR", # final model is also GBR
        n_bootstrap=100,  # Using fewer for speed
        ci_level=0.95
    )
results_GbrDML["runtime_sec"] = time.perf_counter() - start
eval_GbrDML = evaluate_estimator(
    results_GbrDML,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="GbrDML"
)
start = time.perf_counter()
results_LassoDML = dml_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR", # linear ps model, linear outcome model and lasso cate model (the only model using linear models for the nuisance functions)
        method_cate="lasso",
        ci_level=0.95
    )
results_LassoDML["runtime_sec"] = time.perf_counter() - start
eval_LassoDML = evaluate_estimator(
    results_LassoDML,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="LassoDML"
)
start = time.perf_counter()
results_LinearDML = dml_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR", #auto
        method_cate="linear",
        ci_level=0.95
    )
results_LinearDML["runtime_sec"] = time.perf_counter() - start
eval_LinearDML = evaluate_estimator(
    results_LinearDML,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="LinearDML"
)
start = time.perf_counter()
results_GbrDR = drlearner_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="GBR",
        n_bootstrap=100,  # Using fewer for speed
        ci_level=0.95
    )
results_GbrDR["runtime_sec"] = time.perf_counter() - start
eval_GbrDR = evaluate_estimator(
    results_GbrDR,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="GbrDR"
)
start = time.perf_counter()
results_LassoDR = drlearner_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="lasso", # don't use lasso, as linear is faster without boostrapping, since there are only 18 covariates 
        n_bootstrap=100,  # Using fewer for speed
        ci_level=0.95
    )
results_LassoDR["runtime_sec"] = time.perf_counter() - start
eval_LassoDR = evaluate_estimator(
    results_LassoDR,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="LassoDR"
)
start = time.perf_counter()
results_LinearDR = drlearner_binary(
        df2,
        outcome_col=Y,
        treatment_col=T,
        feature_cols=None,
        method="GBR",
        method_cate="linear", # don't use lasso, as linear is faster without boostrapping, since there are only 18 covariates 
        n_bootstrap=100,  # Using fewer for speed
        ci_level=0.95
    )
results_LinearDR["runtime_sec"] = time.perf_counter() - start
eval_LinearDR = evaluate_estimator(
    results_LinearDR,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="LinearDR"
)
start = time.perf_counter()
results_cf = causal_forest(df, 
    outcome = Y, 
    treatment = T, 
    covariates=None, 
    alpha=0.05, 
    num_trees=2000)
results_cf["runtime_sec"] = time.perf_counter() - start
eval_cf = evaluate_estimator(
    results_cf,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="CausalForest"
)
start = time.perf_counter()
results_bcf = bayesian_causal_forest(
    df,
    outcome = Y,
    treatment = T,
    nburn=500,
    nsim=1000,
    alpha=0.05,
    random_state=36
)
results_bcf["runtime_sec"] = time.perf_counter() - start
eval_bcf = evaluate_estimator(
    results_bcf,
    true_cate=true_cate,
    true_ate=true_ate,
    estimator_name="BayesianCausalForest")

# Compare
comparison_df = compare_estimators([eval_GbrXLearner, 
                                    eval_LassoXLearner,
                                    eval_GbrDML, 
                                    eval_LassoDML,
                                    eval_LinearDML,
                                    eval_GbrDR,
                                    eval_LassoDR,
                                    eval_LinearDR,
                                    eval_cf,
                                    eval_bcf])
print("\n" + comparison_df.to_string(index=False))

RUNNING ALL full-version CATE ESTIMATORS WITH MCRPC SYNTHETIC DATA

Dataset shape: (4098, 25)
Treatment prevalence: 0.556
Outcome prevalence: 0.174
True ATE: 0.0538
True CATE range: [-0.0068, 0.1478]

RUNNING ESTIMATORS

EVALUATION METRICS


ERROR:rpy2.rinterface_lib.callbacks:R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xd8 in position 0: invalid continuation byte <traceback object at 0x7fdb3063d980>
ERROR:rpy2.rinterface_lib.callbacks:R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xd8 in position 0: invalid continuation byte <traceback object at 0x7fdb8c7b4100>
ERROR:rpy2.rinterface_lib.callbacks:R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xd8 in position 0: invalid continuation byte <traceback object at 0x7fdb6c7e04c0>
ERROR:rpy2.rinterface_lib.callbacks:R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xd8 in position 0: invalid continuation byte <traceback object at 0x7fdb48602200>
ERROR:rpy2.rinterface_lib.callbacks:R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xd8 in position 0: invalid continuation byte <traceb


           estimator  runtime_sec        ate_bias  ate_stderr  ate_coverage  cate_rmse  cate_coverage
        XLearner-GBR   161.688186  [-0.021300148]    0.083529             1   0.099963       0.819424
      XLearner-Lasso   127.596813  [-0.026548054]    0.034335             1   0.034964       0.799658
              GbrDML   271.488066  [0.0020698644]    0.114073             1   0.132757       0.780137
            LassoDML   239.247268  [0.0024935864]    0.007935             1   0.021368       0.407760
           LinearDML     2.260693   [0.002164945]    0.008533             1   0.041489       0.931186
               GbrDR   275.162003  [0.0073779263]    0.122643             1   0.143988       0.777940
             LassoDR   241.553069  [0.0073779263]    0.008181             1   0.022589       0.331869
            LinearDR     2.268973  [0.0073779263]    0.042284             1   0.043628       0.953880
        CausalForest    21.715634 [-0.0004335642]    0.008781             1   0.0